## 16. 流式输入：用 async generator 驱动会话

> 来源：[Streaming Input](https://code.claude.com/docs/en/agent-sdk/streaming-vs-single-mode)

`prompt` 传 async generator 就进入 **streaming input mode**（官方**默认并推荐**的形态）：agent 作为长生命周期进程运行，一条一条消费 generator yield 出来的消息。比起单条字符串，这种模式多出五项能力：消息里能附带图片、多条消息按顺序排队处理、会话中能完整调用全部工具和自定义 MCP server、边生成边看到实时反馈、多轮之间自动保持 context。另有两个常用能力也只有在这种模式下才成立：实时打断（§15「多轮与打断」）和权限回调（§9.3「can_use_tool 回调」——它只在 streaming input mode 下才会触发）。

消息 dict 格式固定：

```python
{"type": "user", "message": {"role": "user", "content": <字符串 或 content block 列表>}}
```

`content` 传列表时可混合文本与图片 block：

```python
{"type": "user", "message": {"role": "user", "content": [
    {"type": "text", "text": "Review this architecture diagram"},
    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_base64}},
]}}
```

generator 里可以 `await`（等外部条件、等用户下一条输入），yield 之间的时间 agent 在处理已收到的消息——这就是构建 chat 界面的骨架。

> [!warning] Python 专属坑：generator 异常静默吞掉
> generator 内抛异常时，Python SDK 只在 debug 级别记日志，**session 静默卡住、不 raise**。流式会话挂住无输出时，先开 debug logging 检查自己的 generator。

In [ ]:
import asyncio
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
)


async def demo_streaming_input():
    async def message_generator():
        yield {
            "type": "user",
            "message": {"role": "user", "content": "Analyze this project structure"},
        }
        await asyncio.sleep(2)  # 模拟等待外部条件/用户输入
        yield {
            "type": "user",
            "message": {"role": "user", "content": "Now summarize it in one sentence"},
        }

    options = ClaudeAgentOptions(max_turns=10, allowed_tools=["Read", "Grep", "Glob"])
    async with ClaudeSDKClient(options) as client:
        await client.query(message_generator())  # 发送流式输入
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(block.text)


await demo_streaming_input()